# Decision Tree là gì?

- Hãy tưởng tượng ta đang vẽ một sơ đồ ra quyết định. Mọi thứ bắt đầu từ **một câu hỏi duy nhất** — ví dụ, mỗi khi chuẩn bị ra khỏi nhà, ta thường phân vân có nên mang ô theo hay không. Để trả lời, ta đặt câu hỏi: "Trời có đang mưa không?". Ứng với mỗi câu trả lời "có" hoặc "không", sơ đồ sẽ **rẽ sang một nhánh khác nhau**, kèm theo một câu hỏi mới (nếu cần) — cứ như vậy cho đến khi ta chốt được quyết định cuối cùng (mang ô hay không).

- **Decision Tree (Cây quyết định)** chính là mô hình máy học mô phỏng lại quy trình "hỏi - rẽ nhánh - quyết định" đó. Tùy vào việc đầu ra (kết quả cuối cùng) là gì, ta chia Decision Tree thành hai loại:
  - Nếu đầu ra là một **nhãn rời rạc** (ví dụ: Có/Không, Đậu/Rớt), ta gọi là **Classification Tree (Cây phân loại)**.
  - Nếu đầu ra là một **giá trị số liên tục** (ví dụ: giá nhà, điểm số), ta gọi là **Regression Tree (Cây hồi quy)**.

- Trong bài này, chúng ta sẽ tập trung tìm hiểu về **Classification Tree**.

<p align="center">
  <img src="../Images/Decision-tree.png" alt="Decision-tree"/>
</p>

- Một điểm mạnh của Decision Tree là có thể xử lý đồng thời nhiều dạng dữ liệu đầu vào khác nhau:
  - **Dữ liệu rời rạc (Categorical data)**: Là các biến chỉ nhận một số hữu hạn giá trị, ví dụ **Đúng/Sai**, **Trời mưa/Trời nắng**. Với loại dữ liệu này, cây sẽ tạo một nhánh riêng cho từng giá trị.

  - **Dữ liệu liên tục (Continuous/Numeric data)**: Là các biến số thực, ví dụ **Tuổi**, **Điểm số**. Với loại dữ liệu này, thuật toán sẽ tự tìm ra một **ngưỡng (threshold)** phù hợp để chia dữ liệu thành hai nhóm (ví dụ: Tuổi ≤ 30 và Tuổi > 30).

  - **Dữ liệu hỗn hợp (Mixed data)**: Là trường hợp tập dữ liệu có cả biến rời rạc lẫn biến liên tục — Decision Tree có thể xử lý kết hợp cả hai loại này trong cùng một cây.

## 1. Bài toán cần giải quyết (Split Condition)

- Khi xây dựng **Decision Tree**, tại mỗi **nút (node)**, thuật toán phải trả lời câu hỏi:
  > Trong số hàng chục **đặc trưng (features)** của dữ liệu, đâu là feature tốt nhất để dùng làm điều kiện phân chia nhánh (split condition) tại nút này?

- Nói cách khác, máy tính cần một cách **so sánh và chấm điểm** các feature, rồi chọn ra feature "tốt nhất". Để làm được điều đó một cách định lượng (bằng con số cụ thể, chứ không phải cảm tính), ta cần một thước đo toán học — đó chính là **Entropy**, khái niệm nền tảng để tính ra **Information Gain** (mức độ "thông tin" mà một feature mang lại khi dùng để chia nhánh).

- Trong phần dưới đây, ta sẽ tìm hiểu kỹ về **Entropy** trước — đây là viên gạch nền tảng, cần nắm vững trước khi có thể hiểu Information Gain.

## 2. Entropy — Thước đo độ "xáo trộn" của dữ liệu

- Khái niệm **Entropy** được Claude Shannon đưa ra năm 1948 trong lý thuyết Thông tin (Information Theory), dùng để đo mức độ **không chắc chắn (uncertainty)** hoặc **độ xáo trộn/không thuần khiết (impurity)** của một tập dữ liệu.

> *(Information Gain — cách dùng Entropy để chọn ra feature tốt nhất — sẽ được trình bày ở phần tiếp theo của series. Ở phần này, ta tập trung xây dựng thật vững khái niệm Entropy trước.)*

- Trước khi đến với Entropy, ta cần hiểu khái niệm **độ ngạc nhiên (surprise)** của một **sự kiện (event)** $E$. Xét ví dụ sau:
  - Một chiếc túi có 10 viên bi: **9 viên bi đỏ** và **1 viên bi xanh**.
  - Gọi $E_{đỏ}$ là sự kiện "rút ngẫu nhiên được bi đỏ", $E_{xanh}$ là sự kiện "rút ngẫu nhiên được bi xanh".
  - Xác suất tương ứng: $P(E_{đỏ}) = \frac{9}{10} = 0.9$ và $P(E_{xanh}) = \frac{1}{10} = 0.1$.

- Vì bi xanh hiếm hơn bi đỏ ($P(E_{xanh}) < P(E_{đỏ})$), nếu rút trúng bi xanh ta sẽ cảm thấy **bất ngờ hơn** so với rút trúng bi đỏ. Vậy làm thế nào để đo lường "độ bất ngờ" này bằng một con số cụ thể?

- **Đề xuất ban đầu**: $$\text{Surprise}(E) = \frac{1}{P(E)}$$

- Công thức này thỏa được điều kiện mong muốn đầu tiên:
  - Sự kiện càng hiếm $P(E) \downarrow$ thì độ ngạc nhiên lại càng cao $\text{Surprise}(E) \uparrow$.
  - Tuy nhiên, nó lại gặp phải hai vấn đề:
    1. **Đơn vị mơ hồ**: $P(E) = 0.1 \Rightarrow \text{Surprise} = 10$. Vậy **10** này là gì? Ta không thể nói là "10 ngạc nhiên".
    2. **Không cộng dồn được**: Với hai sự kiện độc lập $E_1, E_2$:
        - **Kỳ vọng ban đầu (mong muốn có "tính cộng dồn")**: Lượng bất ngờ khi biết cả hai biến cố cùng xảy ra phải bằng tổng lượng bất ngờ của từng biến cố riêng lẻ hợp lại.

        $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) + \text{Surprise}(E_2)$$

        - **Nhưng thực tế**:
          - Theo xác suất, vì $E_1$ và $E_2$ độc lập nên:

            $$P(E_1 \cap E_2) = P(E_1) \times P(E_2)$$

          - Áp dụng công thức $\text{Surprise}(E) = \frac{1}{P(E)}$ vào $E_1 \cap E_2$:

          $$\text{Surprise}(E_1 \cap E_2) = \frac{1}{P(E_1 \cap E_2)} = \frac{1}{P(E_1) \times P(E_2)}$$

          - Tách phân số ra:

          $$\frac{1}{P(E_1) \times P(E_2)} = \frac{1}{P(E_1)} \times \frac{1}{P(E_2)}$$

          - Kết quả thực tế thu được là: $$\text{Surprise}(E_1 \cap E_2) = \text{Surprise}(E_1) \times \text{Surprise}(E_2)$$

        - Chúng ta mong muốn là **cộng** nhưng thực tế lại là **nhân**.

- Để giải quyết cả hai vấn đề trên, các nhà toán học (điển hình là Claude Shannon) đã chèn thêm hàm **Logarithm** vào công thức tính độ ngạc nhiên, và gọi kết quả là $I(p)$.
  - Sở dĩ Logarithm giải quyết được vấn đề "cộng dồn" là nhờ một tính chất quen thuộc: $\log_2(x \times y) = \log_2(x) + \log_2(y)$ — nghĩa là logarithm biến **phép nhân thành phép cộng**. Đây chính xác là điều ta cần.
- $I(p)$ chính là ký hiệu toán học đại diện cho Surprise(E) (độ bất ngờ / lượng thông tin) của một biến cố $E$ có xác suất xuất hiện là $p$.

- Trong **Lý thuyết thông tin (Information Theory)**:
  - $I$ là viết tắt của **Information (Thông tin)** hoặc **Self-Information (Tự thông tin)**.
  - $p$ là xác suất $P(E)$ của biến cố đó ($0 \le p \le 1$). Thay vì viết $I(P(E))$, người ta viết gọn lại thành $I(p)$.

  $$\text{Surprise}(E) = I(P(E)) = I(p) = \log_2\left(\frac{1}{P(E)}\right) = -\log_2(P(E))$$

- Nhờ tính chất trên, với hai sự kiện độc lập $E_1, E_2$, ta có được đúng tính cộng dồn mà ban đầu ta mong muốn:
$$I(E_1 \cap E_2) = -\log_2 [P(E_1)P(E_2)] = -\log_2 P(E_1) - \log_2 P(E_2) = I(E_1) + I(E_2)$$

### 2.1. Bản chất cốt lõi: Entropy $H(S)$ thực chất là gì?

- **Entropy $H(S)$** chính là **giá trị kỳ vọng (trung bình có trọng số)** của độ bất ngờ $I(p_c)$, tính trên tất cả các nhãn $c$ có trong tập dữ liệu $S$.
- **Entropy** càng lớn thì tập dữ liệu càng "không thuần" (impure), càng khó phân tách.
  > Nhắc lại: $I(p_c)$ ở đây chính là công thức $I(p) = -\log_2(p)$ đã học ở phần trên — chỉ khác là thay vì tính cho một sự kiện $E$ bất kỳ, ta áp dụng cho từng **nhãn (label) $c$**, với $p_c$ là xác suất xuất hiện của nhãn đó.

  $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c \quad (\text{đơn vị: bit})$$

- Trong đó:
  - $S$ (Dataset): Tập dữ liệu cần đo độ hỗn loạn/độ xáo trộn.
  - $\mathcal{C}$ (Classes): Tập hợp tất cả các nhãn phân loại có thể có trong bài toán (Ví dụ: $\mathcal{C} = \{\text{Chơi}, \text{Không chơi}\}$ hoặc $\mathcal{C} = \{0, 1\}$).
  - $c$: Một nhãn phân loại cụ thể thuộc $\mathcal{C}$.
  - $p_c$: Xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$.
  - $-\log_2 p_c$: Chính là $I(p_c)$ — lượng thông tin hay độ ngạc nhiên (Surprise) thu được khi xuất hiện nhãn $c$.
  - **Dấu $\sum$ (Tổng)**: Lấy tổng độ ngạc nhiên của từng nhãn.
  - **Đơn vị bit**: Do sử dụng hàm logarithm cơ số 2 ($\log_2$), đơn vị đo thông tin thu được tính bằng **bit**.

    

### 2.2. Ví dụ minh họa bằng số cụ thể về Entropy $H(S)$

- Giả sử bạn có tập dữ liệu $S$ gồm 10 mẫu phân loại thành 2 nhãn $\{A, B\}$:

#### 2.2.1. Trường hợp 1: Tập dữ liệu hoàn toàn tinh khiết (Pure)

- Gồm **10** mẫu nhãn **A**: $\rightarrow p_A = \frac{10}{10} = 1$
- **0** mẫu nhãn **B**:  $\rightarrow p_B = \frac{0}{10} = 0$.

$$H(S) = -\left(1 \cdot \log_2(1) + 0 \cdot \log_2(0)\right) = -(0 + 0) = 0 \text{ bit}$$

> **Lưu ý nhỏ**: Về mặt toán học, $\log_2(0)$ thực ra không xác định (tiến tới $-\infty$). Tuy nhiên, theo quy ước trong lý thuyết thông tin, ta luôn coi $0 \cdot \log_2(0) = 0$ — vì một nhãn không bao giờ xuất hiện thì không đóng góp gì vào độ "xáo trộn" của tập dữ liệu. Quy ước này áp dụng xuyên suốt cho mọi công thức Entropy về sau.

- **Ý nghĩa**: Entropy = 0 $\rightarrow$ Tập dữ liệu sạch tuyệt đối, không có sự xáo trộn hay mơ hồ nào

#### 2.2.2. Trường hợp 2: Tập dữ liệu xáo trộn tối đa (Impure / Uncertain)

- Gồm **5** mẫu nhãn **A**: $\rightarrow p_A = \frac{5}{10} = 0.5$
- **5** mẫu nhãn **B**:  $\rightarrow p_B = \frac{5}{10} = 0.5$.

$$H(S) = -\left(0.5 \cdot \log_2(0.5) + 0.5 \cdot \log_2(0.5)\right) = -\left(0.5 \cdot (-1) + 0.5 \cdot (-1)\right) = 1 \text{ bit}$$

- **Ý nghĩa**: Entropy đạt giá trị cực đại (= 1 với bài toán 2 nhãn) $\rightarrow$ Tập dữ liệu vô cùng xáo trộn, dự đoán hoàn toàn ngẫu nhiên


#### 2.2.3. Trường hợp 3: Bài toán có 3 nhãn (ví dụ: $A, B, C$)

- Giả sử tập dữ liệu $S$ có **10 mẫu**, được phân thành 3 nhãn $A, B, C$ với số lượng lần lượt là:
  - Nhãn $A$: 5 mẫu
  - Nhãn $B$: 3 mẫu
  - Nhãn $C$: 2 mẫu

  

- $S$ (Dataset): Là toàn bộ tập dữ liệu gồm **10 mẫu** mà ta đang xét.
- $\mathcal{C}$ (Classes): Là tập hợp tất cả các nhãn phân loại có trong tập $S$, cụ thể ở đây $\mathcal{C} = \{A, B, C\}$
- $c$: Là một nhãn cụ thể thuộc tập $\mathcal{C}$. Trong Trường hợp 3, $c$ sẽ lần lượt nhận từng giá trị là $A$, $B$, hoặc $C$ khi lấy tổng.
- $p_c$: Là xác suất xuất hiện của nhãn $c$ trong tập dữ liệu $S$. Cụ thể:
$$p_c = \frac{\text{Số phần tử có nhãn } c}{\text{Tổng số phần tử trong tập } S}$$
  - Khi $c = A \rightarrow p_A = \frac{5}{10} = 0.5$
  - Khi $c = B \rightarrow p_B = \frac{3}{10} = 0.3$
  - Khi $c = C \rightarrow p_C = \frac{2}{10} = 0.2$

- $-\log_2 p_c$: Là **độ ngạc nhiên (Surprise) / lượng thông tin $I(p_c)$** thu được nếu rút ngẫu nhiên được một mẫu mang nhãn $c$:
  - Khi $c = A \rightarrow  I(p_A) = -\log_2(0.5) = 1 \text{ bit}$
  - Khi $c = B \rightarrow  I(p_B) = -\log_2(0.3) \approx 1.73696 \text{ bit}$
  - Khi $c = C \rightarrow  I(p_C) = -\log_2(0.2) \approx 2.32193 \text{ bit}$


- Tính Entropy $H(S)$ bằng cách khai triển đầy đủ từng phần tử
  $$H(S) = \left[ p_A \times I(p_A) \right] + \left[ p_B \times I(p_B) \right] + \left[ p_C \times I(p_C) \right]$$

- Thay các giá trị vừa tính vào:
  $$H(S) = (0.5 \times 1) + (0.3 \times 1.73696) + (0.2 \times 2.32193)$$$$H(S) = 0.5 + 0.521088 + 0.464386$$$$H(S) \approx 1.48547 \text{ bit}$$

#### Một vài tính chất tổng quát của Entropy

- Từ các ví dụ trên, ta có thể rút ra một số tính chất tổng quát sau của Entropy:
-  $H(S)=0$ khi S chỉ có 1 lớp, được gọi là thuần (Pure)
-  $H(S)=1$ đạt cực đại khi $S$ có 2 nhãn phân phối đều (phân loại nhị phân).
- Với $C >2$, tức nhiều hơn 2 nhãn mà dữ liệu phân phối đều thì $H(S) = log2|C|$:
  - $\vert{}\mathcal{C}\vert{}$ (hoặc $K$): Số lượng nhãn phân loại trong tập dữ liệu (ví dụ: có 3 nhãn thì $\vert{}\mathcal{C}\vert{} = 3$, có 4 nhãn thì $\vert{}\mathcal{C}\vert{} = 4$)
  - Dữ liệu phân phối đều: Tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có số lượng bằng nhau, tức là cơ hội xuất hiện của mỗi nhãn là như nhau.
  
  $$p_c = \frac{1}{\vert{}\mathcal{C}\vert{}} \quad \text{cho mọi nhãn } c$$

- Thay vào: $$H(S) = -\sum_{c \in \mathcal{C}} p_c \log_2 p_c$$
  - Vì tất cả $\vert{}\mathcal{C}\vert{}$ nhãn đều có $p_c = \frac{1}{\vert{}\mathcal{C}\vert{}}$, ta thay giá trị này vào công thức:

    1. Thay $p_c$ vào:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) \right)$$
    
    2. Áp dụng tính chất **Logarithm** $\log_2\left(\frac{1}{x}\right) = -\log_2(x)$:
    
      $$\log_2\left(\frac{1}{\vert{}\mathcal{C}\vert{}}\right) = -\log_2(\vert{}\mathcal{C}\vert{})$$
    
    3. Thay ngược lại vào tổng:
      
      $$H(S) = -\sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \left(-\log_2(\vert{}\mathcal{C}\vert{})\right) \right)$$
      
      $$H(S) = \sum_{c \in \mathcal{C}} \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
    
    4. Lấy tổng của $\vert{}\mathcal{C}\vert{}$ phần tử giống hệt nhau: Do ta đang cộng đại lượng $\left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$ đúng $\vert{}\mathcal{C}\vert{}$ lần:
    
      $$H(S) = \vert{}\mathcal{C}\vert{} \times \left( \frac{1}{\vert{}\mathcal{C}\vert{}} \cdot \log_2(\vert{}\mathcal{C}\vert{}) \right)$$
      
    5. Rút gọn $\vert{}\mathcal{C}\vert{}$, ta còn lại:

      $$H(S) = \log_2(\vert{}\mathcal{C}\vert{})$$

- Nếu có 2 nhãn phân phối đều ($p_1 = p_2 = 0.5$):$$H(S) = \log_2(2) = 1 \text{ bit}$$

- Nếu có 3 nhãn phân phối đều ($p_1 = p_2 = p_3 = \frac{1}{3}$):$$H(S) = \log_2(3) \approx 1.585 \text{ bit}$$

- Nếu có 4 nhãn phân phối đều ($p_1 = p_2 = p_3 = p_4 = 0.25$):$$H(S) = \log_2(4) = 2 \text{ bit}$$

  > **Đây chính là Entropy cực đại ($H_{\max}$)**

#### 2.2.4. Đồ thị của hàm Entropy:
- Đồ thị của hàm **Entropy** mô tả mối quan hệ giữa xác suất xuất hiện của nhãn ($p$) và độ xáo trộn/bất định ($H(S)$).Để dễ hình dung nhất, ta xét đồ thị **Entropy** cho bài toán nhị phân (2 nhãn $A$ và $B$), với xác suất xuất hiện nhãn $A$ là $p$, và nhãn $B$ là $1 - p$.

- **Công thức đường cong Entropy nhị phân**:

    $$H(p) = -p \log_2(p) - (1-p) \log_2(1-p)$$

  - **Trục hoành (Trục X)**: Biểu diễn xác suất $p$ (chạy từ $0.0$ đến $1.0$).
  - **Trục tung (Trục Y)**: Biểu diễn giá trị Entropy $H(p)$ tính bằng bit (chạy từ $0.0$ đến $1.0$)


<p align="center">
  <img src="../Images/e0cbfe2d-9702-427d-aaf9-a142635062ec.png" alt="e0cbfe2d-9702-427d-aaf9-a142635062ec"/>
</p>

- **Các điểm quan trọng trên đường cong:**
  - Tại $p = 0$ ($0\%$ mẫu nhãn $A$, $100\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 1$ ($100\%$ mẫu nhãn $A$, $0\%$ mẫu nhãn $B$):
    - $H(p) = 0\text{ bit}$
    - Tập dữ liệu thuần khiết tuyệt đối, không có sự xáo trộn nào.
  - Tại $p = 0.5$ ($50\%$ mẫu nhãn $A$, $50\%$ mẫu nhãn $B$):
    - $H(p) = 1\text{ bit}$ (Đỉnh của đường cong)
    - Dữ liệu phân phối đều 100%, độ ngạc nhiên/không chắc chắn đạt mức tối đa.

---

- **Tóm lại**: Entropy $H(S)$ cho ta biết một tập dữ liệu (hay một nút bất kỳ trong cây) đang "thuần" hay đang "xáo trộn" tới mức nào — Entropy càng thấp, dữ liệu càng thuần; Entropy càng cao, dữ liệu càng lẫn lộn giữa các nhãn.
- Ở phần tiếp theo của series, ta sẽ dùng chính đại lượng Entropy này để xây dựng **Information Gain** — tiêu chí giúp Decision Tree so sánh các feature và chọn ra feature tốt nhất để phân chia tại mỗi nút.